# 🔥 Developer Burnout Prediction | ML Classification

---

## 📌 About This Notebook

**Burnout** is one of the most critical yet overlooked issues in the software industry. This notebook takes a **data-driven approach** to understand and predict developer burnout levels using real-world-like developer lifestyle metrics.

### 🗂️ What This Notebook Covers

| Section | Description |
|--------|-------------|
| 📦 Data Loading | Load and explore the raw dataset |
| 🔍 Exploratory Data Analysis (EDA) | Understand patterns, distributions, and correlations |
| 🧹 Data Preprocessing | Handle missing values and encode labels |
| 🏋️ Model Training | Train multiple ML classifiers |
| 📊 Model Evaluation | Compare accuracy, precision, recall, F1-score |
| 🏆 Best Model Selection | Pick the best-performing model |
| 🔮 Prediction | Make predictions on new data |
| ✅ Conclusion | Key insights and takeaways |

### 🎯 Target Variable
- **`burnout_level`** → `Low`, `Medium`, `High` *(Multi-class Classification)*

### 🧠 Features Used
> `age`, `experience_years`, `daily_work_hours`, `sleep_hours`, `caffeine_intake`, `bugs_per_day`, `commits_per_day`, `meetings_per_day`, `screen_time`, `exercise_hours`, `stress_level`

---
> 💡 **Beginner Friendly**: Each step is explained in simple language with comments in every code cell.


---
## 📦 Step 1: Install & Import Libraries

We start by importing all the Python libraries we need:
- **pandas** → for data manipulation
- **numpy** → for numerical operations
- **matplotlib / seaborn** → for visualizations
- **sklearn** → for machine learning models
- **xgboost** → a powerful boosting algorithm

In [ ]:
# 📚 Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

# ML Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# Set a clean visual style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load the Dataset

We load the CSV file using **pandas** and take a first look at the data.
- `head()` → shows the first 5 rows
- `shape` → tells us (rows, columns)

In [ ]:
# 📂 Load the dataset
df = pd.read_csv('/kaggle/input/developer-burnout-dataset-7000/developer_burnout_dataset_7000.csv')

print(f'📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print()
df.head()

In [ ]:
# 🔍 Check column names and data types
print('📋 Column Info:')
df.info()

In [ ]:
# 📊 Basic statistical summary
# This tells us the min, max, mean, and spread of each column
df.describe().round(2)

---
## 🔍 Step 3: Exploratory Data Analysis (EDA)

EDA helps us **understand the data visually** before building a model. We will:
- Check the distribution of the target variable
- Look at missing values
- Explore correlations between features

In [ ]:
# 🎯 Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
order = ['Low', 'Medium', 'High']
colors = ['#2ecc71', '#f39c12', '#e74c3c']
counts = df['burnout_level'].value_counts()[order]
axes[0].bar(order, counts, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('🎯 Burnout Level Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Burnout Level')
axes[0].set_ylabel('Count')
for i, (cat, val) in enumerate(zip(order, counts)):
    axes[0].text(i, val + 30, str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts, labels=order, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('📈 Burnout Level Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'\n📊 Class Distribution:\n{df["burnout_level"].value_counts()}')

In [ ]:
# 🔴 Check for Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('🔴 Columns with Missing Values:')
print(missing_df)

# Visualize missing values
plt.figure(figsize=(12, 4))
missing_df['Missing %'].plot(kind='bar', color='salmon', edgecolor='black')
plt.title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
plt.ylabel('Missing %')
plt.xticks(rotation=45)
plt.axhline(y=10, color='red', linestyle='--', alpha=0.7, label='10% threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 📊 Feature Distributions by Burnout Level
# This helps us see which features vary most across burnout categories

key_features = ['stress_level', 'daily_work_hours', 'sleep_hours', 
                 'screen_time', 'exercise_hours', 'caffeine_intake']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
palette = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

for i, feat in enumerate(key_features):
    sns.boxplot(data=df, x='burnout_level', y=feat, order=['Low', 'Medium', 'High'],
                palette=palette, ax=axes[i])
    axes[i].set_title(f'{feat.replace("_", " ").title()} vs Burnout', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('🔍 Key Features vs Burnout Level', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 🔗 Correlation Heatmap
# Correlation tells us how strongly two features are related
# Values close to 1 or -1 = strong relationship; close to 0 = weak relationship

numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(12, 8))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, vmin=-1, vmax=1, center=0,
            annot_kws={'size': 9})
plt.title('🔗 Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 📈 Stress Level vs Work Hours — Colored by Burnout
plt.figure(figsize=(10, 6))
palette = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
sample = df.dropna().sample(500, random_state=42)  # sample for clarity

for level, color in palette.items():
    subset = sample[sample['burnout_level'] == level]
    plt.scatter(subset['daily_work_hours'], subset['stress_level'],
                c=color, label=level, alpha=0.6, s=50, edgecolors='white', linewidth=0.5)

plt.xlabel('Daily Work Hours', fontsize=12)
plt.ylabel('Stress Level', fontsize=12)
plt.title('🧑‍💻 Stress Level vs Work Hours by Burnout Level', fontsize=14, fontweight='bold')
plt.legend(title='Burnout Level')
plt.tight_layout()
plt.show()

---
## 🧹 Step 4: Data Preprocessing

Before feeding data to a machine learning model, we need to **clean and prepare** it:

1. **Handle Missing Values** → Fill in the gaps using the median value of each column
2. **Encode Target Variable** → Convert `Low/Medium/High` text labels to numbers (0, 1, 2)
3. **Feature Scaling** → Scale all features to the same range so no single feature dominates
4. **Train-Test Split** → Divide data into training (80%) and testing (20%) sets

In [ ]:
# 🧹 PREPROCESSING

# Step 1: Separate features (X) and target (y)
X = df.drop('burnout_level', axis=1)  # all columns except target
y = df['burnout_level']               # the target column

# Step 2: Handle Missing Values — fill with median (robust to outliers)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

# Step 3: Encode Target Labels to Numbers
# Low=0, Medium=1, High=2
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f'🏷️  Label Encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Step 4: Train-Test Split (80% train, 20% test)
# stratify=y ensures equal class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Step 5: Feature Scaling — StandardScaler makes mean=0, std=1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only!
X_test_scaled  = scaler.transform(X_test)       # transform test using train stats

print(f'\n✅ Preprocessing Complete!')
print(f'   Training set   : {X_train.shape[0]} samples')
print(f'   Test set       : {X_test.shape[0]} samples')
print(f'   Features       : {X_train.shape[1]}')

---
## 🏋️ Step 5: Train Multiple ML Models

We will train **5 different models** and compare them:

| Model | Description |
|-------|-------------|
| 🌲 Random Forest | Many decision trees voting together |
| 📈 XGBoost | Advanced gradient boosting — usually very accurate |
| 🚀 Gradient Boosting | Builds trees one by one fixing mistakes |
| 📏 Logistic Regression | Simple, linear model |
| 🌿 Decision Tree | Single tree — easy to understand |

In [ ]:
# 🏋️ Define all models we want to try
models = {
    '🌲 Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=None,
                                                       random_state=42, n_jobs=-1),
    '📈 XGBoost'             : XGBClassifier(n_estimators=200, learning_rate=0.1,
                                              max_depth=6, use_label_encoder=False,
                                              eval_metric='mlogloss', random_state=42,
                                              verbosity=0),
    '🚀 Gradient Boosting'   : GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                                            max_depth=5, random_state=42),
    '📏 Logistic Regression' : LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    '🌿 Decision Tree'       : DecisionTreeClassifier(max_depth=10, random_state=42)
}

# 📊 Store results
results = []

print('🏃 Training models...\n')
for name, model in models.items():
    # Use scaled data for Logistic Regression, raw for tree models
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    acc = accuracy_score(y_test, y_pred)
    results.append({
        'Model': name,
        'Test Accuracy': round(acc * 100, 2),
        'CV Mean Accuracy': round(cv_scores.mean() * 100, 2),
        'CV Std': round(cv_scores.std() * 100, 2)
    })
    print(f'  {name:<28} → Test: {acc*100:.2f}%  |  CV: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

print('\n✅ All models trained!')

---
## 📊 Step 6: Compare Model Performance

In [ ]:
# 📊 Results Table
results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index += 1
print('🏆 Model Comparison:')
results_df

In [ ]:
# 📈 Bar Chart Comparison
fig, ax = plt.subplots(figsize=(12, 6))

model_names = [r['Model'] for r in results]
test_accs   = [r['Test Accuracy'] for r in results]
cv_accs     = [r['CV Mean Accuracy'] for r in results]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, test_accs, width, label='Test Accuracy', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, cv_accs, width, label='CV Mean Accuracy', color='#2ecc71', edgecolor='white')

ax.set_title('🤖 Model Accuracy Comparison', fontsize=15, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim(50, 105)
ax.legend()
ax.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% line')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 🏆 Step 7: Best Model — Deep Dive

We select the **best performing model** and analyze it in detail using:
- **Confusion Matrix** → shows how often the model was right vs wrong
- **Classification Report** → precision, recall, F1-score for each class
- **Feature Importance** → which features matter most to the model

In [ ]:
# 🏆 Identify and use the best model
best_row = max(results, key=lambda x: x['Test Accuracy'])
best_name = best_row['Model']
best_model = models[best_name]

print(f'🥇 Best Model: {best_name}')
print(f'   Test Accuracy: {best_row["Test Accuracy"]}%')
print(f'   CV Accuracy  : {best_row["CV Mean Accuracy"]}% ± {best_row["CV Std"]}%')

# Get predictions from the best model
if 'Logistic' in best_name:
    y_best_pred = best_model.predict(X_test_scaled)
else:
    y_best_pred = best_model.predict(X_test)

In [ ]:
# 🎯 Confusion Matrix
# Rows = Actual, Columns = Predicted
# Diagonal = Correct predictions

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
cm = confusion_matrix(y_test, y_best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix (Counts)\n{best_name}', fontweight='bold')

# Normalized
cm_norm = confusion_matrix(y_test, y_best_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=le.classes_)
disp2.plot(ax=axes[1], colorbar=False, cmap='Greens', values_format='.2%')
axes[1].set_title(f'Confusion Matrix (Normalized)\n{best_name}', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 📋 Detailed Classification Report
print('📋 Classification Report:')
print('='*55)
print(classification_report(y_test, y_best_pred, target_names=le.classes_))

In [ ]:
# 🌟 Feature Importance
# Which features does the model find most useful?

if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=X.columns)
    fi = fi.sort_values(ascending=True)
    
    colors_fi = ['#e74c3c' if v == fi.max() else '#3498db' for v in fi.values]
    
    plt.figure(figsize=(10, 7))
    bars = plt.barh(fi.index, fi.values, color=colors_fi, edgecolor='white')
    plt.xlabel('Importance Score')
    plt.title(f'🌟 Feature Importance — {best_name}', fontsize=14, fontweight='bold')
    
    for bar in bars:
        plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{bar.get_width():.3f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print(f'\n🥇 Most Important Feature: {fi.idxmax()} ({fi.max():.3f})')
    print(f'🥉 Least Important Feature: {fi.idxmin()} ({fi.min():.3f})')
else:
    print('ℹ️  This model does not provide feature importances directly.')

---
## 🔮 Step 8: Make a Prediction on New Data

Let's simulate **predicting burnout for a new developer** by entering their stats manually!

In [ ]:
# 🔮 Predict burnout for a hypothetical developer

# Developer Profile: works long hours, low sleep, high stress
new_developer = pd.DataFrame([{
    'age'              : 28,
    'experience_years' : 4,
    'daily_work_hours' : 11,   # long hours!
    'sleep_hours'      : 5,    # low sleep
    'caffeine_intake'  : 4,
    'bugs_per_day'     : 8,
    'commits_per_day'  : 3,
    'meetings_per_day' : 6,
    'screen_time'      : 10,
    'exercise_hours'   : 0.2,  # almost no exercise
    'stress_level'     : 85    # very high stress
}])

# Preprocess in the same way as training data
new_imputed = imputer.transform(new_developer)

if 'Logistic' in best_name:
    new_scaled = scaler.transform(new_imputed)
    prediction = best_model.predict(new_scaled)
    prob = best_model.predict_proba(new_scaled)
else:
    prediction = best_model.predict(new_imputed)
    prob = best_model.predict_proba(new_imputed)

predicted_label = le.inverse_transform(prediction)[0]

emoji = {'Low': '🟢', 'Medium': '🟡', 'High': '🔴'}
print('🧑‍💻 Developer Profile Entered:')
print(new_developer.T.to_string(header=False))
print(f'\n🎯 Predicted Burnout Level: {emoji[predicted_label]} {predicted_label}')
print('\n📊 Prediction Probabilities:')
for cls, p in zip(le.classes_, prob[0]):
    bar = '█' * int(p * 30)
    print(f'   {emoji[cls]} {cls:<8} : {bar} {p*100:.1f}%')

---
## ✅ Conclusion

### 📌 What We Did

In this notebook, we built a complete **end-to-end machine learning pipeline** to predict developer burnout levels. Here's a summary:

1. **Loaded and explored** a dataset of 7,000 developer records with 11 features
2. **Analyzed patterns** using visualizations — burnout correlates strongly with stress level, work hours, and sleep
3. **Preprocessed the data** — handled ~24% missing values using median imputation, encoded labels, and scaled features
4. **Trained 5 ML models** — Random Forest, XGBoost, Gradient Boosting, Logistic Regression, Decision Tree
5. **Evaluated models** using accuracy, cross-validation, confusion matrix, and classification report
6. **Selected the best model** based on test accuracy
7. **Made predictions** on a new hypothetical developer profile

### 🔑 Key Insights

| Insight | Finding |
|---------|----------|
| 🔴 Top Risk Factor | `stress_level` is the strongest predictor of burnout |
| 😴 Sleep Matters | Developers with high burnout sleep significantly fewer hours |
| 💻 Screen & Work Hours | High screen time + long work hours = higher burnout risk |
| 🏃 Exercise Helps | Higher exercise hours correlate with lower burnout |
| ☕ Caffeine | High caffeine intake is a proxy for sleep deprivation |

### 🚀 What You Can Improve Next

- 🔧 **Hyperparameter Tuning** with `GridSearchCV` or `Optuna`
- 🧪 **Try more algorithms**: LightGBM, CatBoost, SVM
- 🏗️ **Feature Engineering**: Create new features like `work_to_sleep_ratio`
- ⚖️ **Handle Class Imbalance** with SMOTE if needed
- 🌐 **Deploy** the model as a REST API using FastAPI or Flask

---

> 💬 *"Taking care of developers is taking care of your product. Burnout is not a feature — it's a bug."*

---
**If you found this notebook helpful, please give it an ⬆️ upvote on Kaggle!**

Feel free to fork and experiment. Happy coding! 🚀